# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산을 파이썬으로 직접 해 본다. 이중차분의 계산은 대부분 평균을 내고 빼는 것이라, 새로 배울 함수는 `groupby`로 건수를 세는 것과 회귀식 하나(`smf.glm`)뿐이다.

1. 데이터 읽기 — 구 × 계약일의 계약 건수
2. 처치와 기간 정하기
3. 주별 건수와 2×2 표 — 표 2-1과 대역
4. 보고서의 그림 — 지정 직전 = 100
5. 회귀식으로 — 상호작용 하나
6. 지정 전 다섯 달과 지정 뒤 몇 주 — 그림 3-1, 기준 기간 바꾸기, 뒤로 밀린 계약
7. 정답이 있는 예 — 일자리 제공 프로그램 자료(표 4-1, 그림 4-1)
8. 더 해 보기: 역할을 바꿔 — 2025년 10월 20일 서울 전역 지정
9. 더 해 보기: 자료 만들기 — 실거래가 원자료에서 1절의 CSV로

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. `as` 뒤의 이름은 줄임말이고, 이후 코드에서는 `pd`, `smf`처럼 줄임말로 부른다. 이 장에서 쓰는 패키지는 모두 Anaconda와 Colab에 기본으로 들어 있다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import numpy as np                       # 지수(np.exp) 계산
import pandas as pd                      # 표(데이터프레임) 다루기
import statsmodels.api as sm             # 회귀식의 분포족(포아송)
import statsmodels.formula.api as smf    # 회귀분석
import matplotlib.pyplot as plt          # 그림

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기

이 장의 자료는 서울 25개 구의 아파트 매매 계약 건수를 계약일마다 센 것이다(2024년 10월 8일 ~ 2026년 10월 6일). 국토교통부 실거래가 공개시스템에서 받은 거래 자료에서 해제된 거래를 빼고, 중개거래와 직거래를 모두 구마다 날마다 셌다. 계약이 없던 날도 0건으로 들어 있다. 책 사이트의 장 첫 페이지에서 내려받을 수 있고, 원자료에서 이 파일을 만드는 코드는 9절에 있다.

- 내 컴퓨터의 주피터: CSV를 이 노트북 파일과 같은 폴더에 두고, 아래 코드의 `CSV =` 줄에 파일 이름을 그대로 적는다.
- Colab: 왼쪽의 폴더 아이콘을 누르고 업로드 버튼으로 CSV를 올린 뒤 `CSV = '/content/파일이름.csv'`로 적는다.

| 변수 | 뜻 |
|---|---|
| `계약일` | 계약한 날(예: 2025-03-24) |
| `구` | 서울의 구 이름 |
| `계약건수` | 그 구에서 그날 맺은 아파트 매매 계약 건수 |

`parse_dates=['계약일']`은 계약일 열을 글자가 아니라 날짜로 읽으라는 옵션이다. 날짜로 읽어야 주를 묶고 기간을 자를 수 있다.

In [ ]:
CSV = 'seoul-apt-contracts-daily-by-gu-2024-10-2026-10.csv'   # 자기 파일 이름으로 바꾼다

df = pd.read_csv(CSV, parse_dates=['계약일'])
print(len(df), '행,', df['구'].nunique(), '개 구,', df['계약일'].min().date(), '~', df['계약일'].max().date())
print('계약', df['계약건수'].sum(), '건')
df.head()

## 2. 처치와 기간 정하기

누가 언제 처치를 받았는지는 자료에 적혀 있지 않으므로 직접 적는다. 2025년 3월 24일부터 강남·서초·송파·용산구 아파트 전체가 토지거래허가구역이 되었다. 이 네 구가 처치 그룹이고, 나머지 21개 구가 비교 상대다.

비교할 두 기간은 보고서와 같게 4주씩 잡는다. 지정 전 4주는 2월 17일 ~ 3월 16일이다. 3월 19일 발표 뒤 계약이 몰린 3월 17일 주는 뺀다(3절의 표에서 확인한다). 지정 뒤 4주는 3월 24일 ~ 4월 20일이다.

주는 월요일에 시작한다. `dt.to_period('W-SUN')`은 일요일에 끝나는 주로 묶고, `.dt.start_time`은 그 주의 첫날(월요일)을 꼬리표로 붙인다. `isin(G4)`는 구 이름이 G4 목록에 들어 있으면 참이고, `astype(int)`가 참·거짓을 1과 0으로 바꾼다.

In [ ]:
G4 = ['강남구', '서초구', '송파구', '용산구']    # 처치 그룹
PRE = ('2025-02-17', '2025-03-16')              # 지정 전 4주
POST = ('2025-03-24', '2025-04-20')             # 지정 뒤 4주

df['4구'] = df['구'].isin(G4).astype(int)
df['주'] = df['계약일'].dt.to_period('W-SUN').dt.start_time   # 월요일에 시작하는 주

## 3. 주별 건수와 2×2 표 — 표 2-1과 대역

구 묶음(4개 구, 21개 구)마다 주별로 계약 건수를 더한다. `groupby(['주', '4구'])['계약건수'].sum()`은 주와 묶음의 짝마다 건수를 더하고, `unstack()`은 묶음을 열로 펼친다. 열 이름 0과 1을 알아보기 쉽게 바꾼다.

먼저 3월 17일 주 앞뒤를 출력해 발표 뒤 계약이 몰린 것을 확인한다. 그다음 두 기간의 주 평균을 내고, 묶음마다 지정 전 4주를 100으로 맞춘다(2절의 "지정 직전 = 100"). 21개 구가 100에서 간 곳이 4개 구의 대역이고, 4개 구의 실제를 대역으로 나눈 것이 "지정이 없었을 때의 몇 분의 1"이다.

마지막 줄은 2절에서 본 "건수를 그대로 빼면"이다. 4개 구도 21개 구만큼 건수가 줄었다고 보면 4개 구의 대역이 주 200건 남짓으로 내려가는데, 이것이 크기가 다른 두 그룹을 건수로 비교할 때의 문제였다.

In [ ]:
wk = df.groupby(['주', '4구'])['계약건수'].sum().unstack().rename(columns={0: '21개 구', 1: '4개 구'})
print(wk.loc['2025-03-03':'2025-04-07'])        # 3월 17일 주에 4개 구 계약이 몰렸다

pre = wk.loc[PRE[0]:PRE[1]].mean()              # 지정 전 4주의 주 평균
post = wk.loc[POST[0]:POST[1]].mean()           # 지정 뒤 4주의 주 평균
idx = post / pre * 100                          # 지정 직전 = 100
table = pd.DataFrame({'지정 전 4주': pre.round(0), '지정 뒤 4주': post.round(0), '지정 직전 = 100': idx.round(0)})
print(table)

stand_in = idx['21개 구']                       # 4개 구의 대역: 21개 구처럼 움직였다면
print(f"4개 구의 대역 100 → {stand_in:.0f}, 실제 {idx['4개 구']:.0f}, 실제 ÷ 대역 {idx['4개 구'] / stand_in:.3f}")
print(f"건수를 그대로 빼면 4개 구의 대역: {pre['4개 구'] - (pre['21개 구'] - post['21개 구']):.0f}건")

## 4. 보고서의 그림 — 지정 직전 = 100

보고서의 그림처럼 2024년 10월 중순부터 2025년 6월 중순까지의 주별 건수를, 묶음마다 지정 전 4주의 평균으로 나눠 100을 곱해 그린다. `axvspan`은 비교한 두 기간을 회색 띠로, `axvline`은 지정일을 세로 점선으로 표시한다.

In [ ]:
w = wk.loc['2024-10-14':'2025-06-15'] / pre * 100

fig, ax = plt.subplots(figsize=(8.4, 3.6))
for lo, hi in (PRE, POST):
    ax.axvspan(pd.Timestamp(lo), pd.Timestamp(hi) + pd.Timedelta(days=6), color='#efeee9')
ax.axvline(pd.Timestamp('2025-03-24'), color='gray', ls='--', lw=0.8)
ax.axhline(100, color='lightgray', lw=0.6)
ax.plot(w.index, w['21개 구'], color='#2a78d6', label='나머지 21개 구')
ax.plot(w.index, w['4개 구'], color='#eb6834', label='강남·서초·송파·용산')
ax.set_ylabel('주별 계약 건수\n(지정 직전 4주 평균 = 100)')
ax.legend(frameon=False)
plt.show()

## 5. 회귀식으로 — 상호작용 하나

보고서의 표 주에 적힌 회귀식이다(2절의 "회귀식으로 쓰면"). 자료를 구 × 주로 만들고, 두 기간의 주만 남긴다. 계약이 없던 구 × 주도 0건으로 남아야 하므로 `unstack(fill_value=0).stack()`으로 빈칸을 0으로 채운다.

회귀식 `건수 ~ C(구) + 후 + 처치:후`는 2절의 식 $	ext{로그 계약 건수}_{it} = a_i + c 	imes 	ext{지정 뒤}_t + d 	imes (	ext{4개 구}_i 	imes 	ext{지정 뒤}_t)$를 코드로 쓴 것이다. `C(구)`는 구마다 자기 높이 $a_i$를 두는 구 더미로, 첫째 빼기에서 지워지는 몫(구마다 원래 몇 건짜리였나)을 맡는다. `후`는 지정 뒤이면 1인 더미로, 계수가 $c$이고 둘째 빼기에서 지워지는 몫(지정 뒤에 모든 구에 함께 일어난 변화)을 맡는다. `처치:후`는 4개 구 더미와 지정 뒤 더미를 곱한 상호작용 항으로, 그 계수 $d$가 두 번 빼기의 답이다. 4개 구 더미를 따로 넣지 않는 것은 구 더미에 이미 들어 있기 때문이다.

종속변수가 건수라서 `smf.ols` 대신 포아송 회귀(`smf.glm(..., family=sm.families.Poisson())`)를 쓴다. 회귀식을 쓰고 계수를 읽는 원리는 다중회귀분석과 같고, 종속변수에 로그를 취한 식이라 계수 $d$를 $e^d$배로 읽는다. 표준오차는 같은 구의 여러 주가 서로 닮은 것을 감안해 구 단위 군집 표준오차로 구한다(`cov_type='cluster'`). 3절의 "실제 ÷ 대역"과 같은 0.105가 나오는 것을 확인한다.

In [ ]:
gw = df.groupby(['구', '주'])['계약건수'].sum().unstack(fill_value=0).stack().rename('건수').reset_index()
in_pre = (gw['주'] >= PRE[0]) & (gw['주'] <= PRE[1])
in_post = (gw['주'] >= POST[0]) & (gw['주'] <= POST[1])
gw = gw[in_pre | in_post].copy()
gw['처치'] = gw['구'].isin(G4).astype(int)
gw['후'] = (gw['주'] >= POST[0]).astype(int)

m = smf.glm('건수 ~ C(구) + 후 + 처치:후', gw, family=sm.families.Poisson()).fit(
    cov_type='cluster', cov_kwds={'groups': gw['구'].astype('category').cat.codes})
b, se = m.params['처치:후'], m.bse['처치:후']
print(f"처치:후 계수 {b:.3f} (표준오차 {se:.3f}), e^b = {np.exp(b):.3f}배, "
      f"95% 신뢰구간 {np.exp(b - 1.96 * se):.3f} ~ {np.exp(b + 1.96 * se):.3f}, 구 × 주 {len(gw)}개")

## 6. 지정 전 다섯 달과 지정 뒤 몇 주 — 그림 3-1, 기준 기간 바꾸기, 뒤로 밀린 계약

3절의 그림 3-1이다. 이번에는 기준을 해제 이전 15주(2024년 10월 14일 ~ 2025년 1월 26일)로 잡아 그 평균을 100으로 맞춘다. 15주 동안 두 선이 거의 겹치다가, 2월 13일 잠실·삼성·대치·청담의 허가구역 해제 뒤로 4개 구만 더 빨리 늘어나는 것을 확인한다. 4개 구의 3월 17일 주는 그림 밖으로 올라가므로 `set_ylim`으로 위를 자른다.

월별로 4개 구 ÷ 21개 구를 계산하면 같은 이야기를 숫자로 볼 수 있다. 마지막으로 3절의 계산을 기준 기간만 그림의 15주로 바꿔 다시 해 본다. 남은 몫이 약 10분의 1에서 약 15%로 바뀌는 것을 확인한다.

끝으로 3절의 "허가를 기다린 계약"이다. 지정 뒤 4개 구의 주별 건수를 출력해 주마다 조금씩 늘어나는 것을 보고, 지정 뒤 기간을 4주, 8주, 12주로 늘려 가며 3절의 계산을 되풀이한다. 지정 전 기간은 그대로 두고 지정 뒤 기간만 늘린다. 12주는 6월 15일까지로, 6월 말의 대출 규제 앞에서 멈춘다.

In [ ]:
base = wk.loc['2024-10-14':'2025-01-26'].mean()
w2 = wk.loc['2024-10-14':'2025-04-27'] / base * 100

fig, ax = plt.subplots(figsize=(8.4, 3.6))
for day in ['2025-02-13', '2025-03-24']:
    ax.axvline(pd.Timestamp(day), color='gray', ls='--', lw=0.8)
ax.axhline(100, color='lightgray', lw=0.6)
ax.plot(w2.index, w2['21개 구'], color='#2a78d6', label='나머지 21개 구')
ax.plot(w2.index, w2['4개 구'], color='#eb6834', label='강남·서초·송파·용산')
ax.set_ylim(0, 450)
ax.set_ylabel('주별 계약 건수\n(해제 이전 15주 평균 = 100)')
ax.legend(frameon=False)
plt.show()

mo = df.groupby([df['계약일'].dt.to_period('M'), '4구'])['계약건수'].sum().unstack()
print('월별 4개 구 ÷ 21개 구:', (mo[1] / mo[0]).loc['2024-10':'2025-03'].round(2).to_dict())

pre2 = wk.loc['2024-10-14':'2025-01-26'].mean()   # 해제 이전 15주를 기준으로
idx2 = post / pre2 * 100
print(f"해제 이전 기준: 21개 구 100 → {idx2['21개 구']:.0f}, 4개 구 100 → {idx2['4개 구']:.0f}, "
      f"실제 ÷ 대역 {idx2['4개 구'] / idx2['21개 구']:.3f}")

print('지정 뒤 4주의 4개 구 주별 건수:', wk.loc[POST[0]:POST[1], '4개 구'].tolist())
for end in ['2025-04-20', '2025-05-18', '2025-06-15']:          # 지정 뒤 4주, 8주, 12주
    idx3 = wk.loc[POST[0]:end].mean() / pre * 100
    print(f"지정 뒤 ~ {end}: 21개 구 100 → {idx3['21개 구']:.0f}, 4개 구 100 → {idx3['4개 구']:.0f}, "
          f"실제 ÷ 대역 {idx3['4개 구'] / idx3['21개 구']:.3f}")

## 7. 정답이 있는 예 — 일자리 제공 프로그램 자료(표 4-1, 그림 4-1)

4절에서 쓴 인과 장의 자료다. 인과 장의 노트북처럼 데헤이아의 누리집에서 바로 읽는다(인터넷에 연결돼 있어야 한다). 열 이름도 같다 — `treat`는 참가자이면 1, `re74`, `re75`, `re78`은 1974년, 1975년, 1978년 소득이다.

표 4-1의 이중차분 열은 종속변수를 1978년 소득 대신 늘어난 소득(`re78 - re75`)으로 바꾸고 `treat` 하나만 넣은 회귀식의 계수다. 비교 상대를 실험 대조군과 일반 인구로 바꿔 가며 계산한다. 회귀식 안에서 `I(re78 - re75)`라고 쓰면 따로 열을 만들지 않아도 된다.

같은 값을 대역으로도 계산한다. 참가자의 1975년 평균 소득에 비교 상대가 늘어난 만큼을 더한 것이 "참가하지 않았다면"의 1978년 소득(대역)이고, 실제 1978년 평균 소득과 대역의 차이가 회귀식의 계수와 같다. 마지막으로 참가 전인 1974년에서 1975년까지 평균 소득이 얼마나 바뀌었는지를 그룹마다 센다(4절의 "나란해 보여도 증명은 아니다").

In [ ]:
URL = 'https://users.nber.org/~rdehejia/data/'
COLS = ['treat', 'age', 'educ', 'black', 'hisp', 'married', 'nodegree', 're74', 're75', 're78']
read = lambda name: pd.read_csv(URL + name + '.txt', sep=r'\s+', header=None, names=COLS)
treated, control, cps = read('nswre74_treated'), read('nswre74_control'), read('cps_controls')

for name, ctrl in [('실험 대조군', control), ('일반 인구(CPS)', cps)]:
    d = pd.concat([treated, ctrl])
    lvl = smf.ols('re78 ~ treat', d).fit()
    did = smf.ols('I(re78 - re75) ~ treat', d).fit()
    print(f"{name:10s} 1978년 소득 비교 {lvl.params['treat']:7,.0f} ({lvl.bse['treat']:,.0f})   "
          f"이중차분 {did.params['treat']:6,.0f} ({did.bse['treat']:,.0f})")

t75, t78 = treated['re75'].mean(), treated['re78'].mean()
for name, g in [('일반 인구만큼', cps), ('실험 대조군만큼', control)]:
    gain = g['re78'].mean() - g['re75'].mean()
    print(f"대역({name} 늘었다면): {t75 + gain:,.0f},  실제 {t78:,.0f}와의 차이 {t78 - t75 - gain:,.0f}")

for name, g in [('참가자', treated), ('실험 대조군', control), ('일반 인구(CPS)', cps)]:
    print(f"{name:10s} 1974 → 1975년 평균 소득의 변화 {g['re75'].mean() - g['re74'].mean():6,.0f}")

fig, ax = plt.subplots(figsize=(7.2, 4.0))
for name, g, col in [('프로그램 참가자', treated, '#eb6834'), ('실험 대조군', control, '#2a78d6'), ('일반 인구(CPS)', cps, 'gray')]:
    ax.plot([1974, 1975, 1978], [g['re74'].mean(), g['re75'].mean(), g['re78'].mean()], marker='o', color=col, label=name)
ax.set_xticks([1974, 1975, 1978])
ax.set_ylabel('평균 소득 (달러)')
ax.legend(frameon=False)
plt.show()

## 8. 더 해 보기: 역할을 바꿔 — 2025년 10월 20일 서울 전역 지정

본문에서는 다루지 않은 계산이다. 2025년 10월 15일 대책으로 10월 20일부터 서울 전역이 토지거래허가구역이 되었다. 강남·서초·송파·용산은 3월부터 이미 지정돼 있었으므로, 이번에는 역할이 뒤바뀐다. 처치 그룹은 21개 구, 비교 상대는 4개 구다.

기간은 6주씩 잡는다. 지정 전은 8월 25일 ~ 10월 5일이다. 10월 6일 주는 추석 연휴가 끼었고, 10월 13일 주는 대책 발표 뒤 계약이 몰렸으므로 둘 다 뺀다. 지정 뒤는 10월 20일 ~ 11월 30일이다. 3절의 계산을 그대로 하되, 대역은 이번에는 4개 구가 100에서 간 곳이다.

21개 구의 계약이 크게 줄어드는 것은 3월의 4개 구와 같다. 그런데 비교 상대인 4개 구의 계약이 오히려 늘어난 것을 확인한다. 같은 대책에 4개 구에도 걸린 다른 내용(대출 규제 등)이 있었는지, 4개 구 쪽에 다른 사정이 있었는지는 이 자료로 가릴 수 없다. 그래서 이 대역이 21개 구의 "지정이 없었다면"을 대신할 만한지는 따로 따져 봐야 한다. 직접 해 보기 1의 "역할을 바꿔 다시 해 보기"에서 AI에게 이 점을 물어보자.

In [ ]:
PRE2 = ('2025-08-25', '2025-10-05')
POST2 = ('2025-10-20', '2025-11-30')
print(wk.loc['2025-09-22':'2025-11-03'])        # 10월 13일 주에 21개 구 계약이 몰렸다

pre_b = wk.loc[PRE2[0]:PRE2[1]].mean()
post_b = wk.loc[POST2[0]:POST2[1]].mean()
idx_b = post_b / pre_b * 100
print(pd.DataFrame({'지정 전 6주': pre_b.round(0), '지정 뒤 6주': post_b.round(0), '지정 직전 = 100': idx_b.round(0)}))
print(f"21개 구의 대역 100 → {idx_b['4개 구']:.0f}, 실제 {idx_b['21개 구']:.0f}, 실제 ÷ 대역 {idx_b['21개 구'] / idx_b['4개 구']:.3f}")

## 9. 더 해 보기: 자료 만들기 — 실거래가 원자료에서 1절의 CSV로

1절의 CSV를 만든 코드다. 국토교통부 실거래가 공개시스템에서 서울 아파트 매매 자료를 1년씩 두 번 받았다(2024년 10월 8일 ~ 2025년 10월 7일, 2025년 10월 8일 ~ 2026년 10월 7일). 받는 방법은 단순회귀분석 장의 직접 해 보기 2, "데이터 수집"과 같다. 두 파일을 이 노트북과 같은 폴더에 두고 `MAKE = True`로 바꿔 실행한다.

원자료는 표 위에 안내문이 있는데, 받은 때에 따라 그 줄 수가 다를 수 있다. 그래서 `"NO"`로 시작하는 머리글 줄을 찾아 그 위를 건너뛴다. 해제된 거래(`해제사유발생일`이 적힌 거래)는 뺀다. `시군구`의 둘째 낱말이 구 이름이다. 마지막으로 구 × 날짜의 모든 짝을 만들어 계약이 없던 날을 0건으로 채운다(`reindex(..., fill_value=0)`).

In [ ]:
MAKE = False   # 원자료 두 파일이 있으면 True로 바꾼다
RAW = ['seoul-apt-sales-2024-10-2025-10.csv', 'seoul-apt-sales-2025-10-2026-10.csv']

def read_raw(path):
    with open(path, encoding='cp949') as f:
        for i, line in enumerate(f):
            if line.startswith('"NO"'):        # 머리글 줄
                break
    return pd.read_csv(path, encoding='cp949', skiprows=i, thousands=',')

if MAKE:
    raw = pd.concat([read_raw(p) for p in RAW], ignore_index=True)
    cancel = raw['해제사유발생일'].astype(str).str.strip()
    raw = raw[cancel.isin(['', '-', 'nan'])].copy()            # 해제된 거래 빼기
    raw['구'] = raw['시군구'].str.split().str[1]
    raw['계약일'] = pd.to_datetime(raw['계약년월'].astype(str) + raw['계약일'].astype(str).str.zfill(2), format='%Y%m%d')
    days = pd.date_range(raw['계약일'].min(), raw['계약일'].max(), freq='D')
    full = pd.MultiIndex.from_product([sorted(raw['구'].unique()), days], names=['구', '계약일'])
    daily = raw.groupby(['구', '계약일']).size().reindex(full, fill_value=0).rename('계약건수').reset_index()
    daily['계약일'] = daily['계약일'].dt.strftime('%Y-%m-%d')
    daily[['계약일', '구', '계약건수']].sort_values(['계약일', '구']).to_csv(CSV, index=False, encoding='utf-8-sig')
    print(len(daily), '행 저장:', CSV)

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 이중차분의 계산은 묶음마다 두 기간의 평균을 내고, 크기를 100으로 맞추고, 비교 상대가 움직인 만큼을 처치 그룹에 빌려 오는 것이 전부였다(3절). 회귀식은 같은 계산을 상호작용 하나로 했다(5절). 계산은 어렵지 않지만, 그 숫자를 효과로 해석할 수 있느냐는 두 그룹이 나란히 움직였으리라는 가정에 달려 있고(6, 7절), 같은 가정 아래서도 처치 뒤 기간을 어떻게 잡느냐에 따라 답이 달라진다(6절).

자기 자료로 분석하려면 1절에서 자기 CSV를 읽고(날짜, 대상, 결과의 세 열), 2절의 `G4`(처치 그룹), `PRE`, `POST`(두 기간)를 바꾼다. 처치 날짜 앞뒤에 결과가 유난히 몰리거나 빠진 주가 있는지 3절의 출력으로 먼저 보고, 있으면 기간에서 뺀다. 6절의 기준 기간과 처치 뒤 기간도 자기 자료의 사정에 맞게 바꿔, 결과가 기간에 따라 얼마나 움직이는지 함께 본다. 7절은 정답이 있는 이 장의 회수 자료라 자기 자료로는 건너뛴다. AI가 만든 보고서를 받았다면, 이 코드로 같은 숫자가 나오는지 확인해 보는 것이 가장 확실한 점검이다.